# CS-GY 9223 — Neuroinformatics
# Project 1: Low-Field to High-Field MRI Enhancement

**Team Members:**
- Nibish Tamrakar (nt2920)
- Tarik Kassa (tk2766)

## Objective
Develop an algorithm that enhances low-field (64mT) brain MRI volumes to approximate 
high-field (3T) image quality. The model takes a low-resolution input of shape 
(112, 138, 40) and produces an enhanced output of shape (179, 221, 200).

## Approach
We use a 3D ResNet that learns the residual (difference) between a bicubic-upsampled 
baseline and the true high-field target. The model is trained on 18 paired volumes 
using patch-based training and 6-fold cross-validation.

In [16]:
import os
# import metric
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.ndimage import zoom, gaussian_filter
from sklearn.model_selection import KFold
from torch.utils.data import Dataset, DataLoader
from extract_slices import load_nifti, create_submission_df, slice_to_base64

## 1. Data Loading & Exploration
We load the NIfTI (.nii) files using nibabel via the provided `extract_slices.py` utility.
Each training pair consists of a low-field volume (112×138×40 voxels at 1.6×1.6×5.0mm) 
and a high-field volume (179×221×200 voxels at 1.0×1.0×1.0mm isotropic).

In [17]:
TRAIN_LF_DIR = "train/low_field"
TRAIN_HF_DIR = "train/high_field"
TEST_DIR = "test"

In [18]:
lf_path = os.path.join(TRAIN_LF_DIR, "sample_001_lowfield.nii")
hf_path = os.path.join(TRAIN_HF_DIR, "sample_001_highfield.nii")

lf_vol = load_nifti(lf_path)
hf_vol = load_nifti(hf_path)

print("Low-field shape :", lf_vol.shape)
print("High-field shape:", hf_vol.shape)
print("Low-field dtype :", lf_vol.dtype)
print("High-field dtype:", hf_vol.dtype)

Low-field shape : (112, 138, 40)
High-field shape: (179, 221, 200)
Low-field dtype : float64
High-field dtype: float64


In [19]:
train_data = {}

for fname in sorted(os.listdir(TRAIN_LF_DIR)):
    if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
        continue

    sample_id = fname.replace(
        "_lowfield.nii.gz", "").replace("_lowfield.nii", "")
    sample_id = sample_id.replace(".nii.gz", "").replace(".nii", "")

    lf_path = os.path.join(TRAIN_LF_DIR, fname)

    # match high_field filename patterns
    hf_candidates = [
        os.path.join(TRAIN_HF_DIR, f"{sample_id}_highfield.nii.gz"),
        os.path.join(TRAIN_HF_DIR, f"{sample_id}_highfield.nii"),
        os.path.join(TRAIN_HF_DIR, f"{sample_id}.nii.gz"),
        os.path.join(TRAIN_HF_DIR, f"{sample_id}.nii"),
    ]
    hf_path = next((p for p in hf_candidates if os.path.exists(p)), None)
    if hf_path is None:
        raise FileNotFoundError(
            f"Missing highfield for {sample_id}. Looked in: {hf_candidates}")

    lf_vol = load_nifti(lf_path)
    hf_vol = load_nifti(hf_path)

    train_data[sample_id] = {"low": lf_vol, "high": hf_vol}

    print(f"{sample_id}: LF {lf_vol.shape}, HF {hf_vol.shape}")

print("\ntotal data loaded:", len(train_data))

sample_001: LF (112, 138, 40), HF (179, 221, 200)
sample_002: LF (112, 138, 40), HF (179, 221, 200)
sample_003: LF (112, 138, 40), HF (179, 221, 200)
sample_004: LF (112, 138, 40), HF (179, 221, 200)
sample_005: LF (112, 138, 40), HF (179, 221, 200)
sample_006: LF (112, 138, 40), HF (179, 221, 200)
sample_007: LF (112, 138, 40), HF (179, 221, 200)
sample_008: LF (112, 138, 40), HF (179, 221, 200)
sample_009: LF (112, 138, 40), HF (179, 221, 200)
sample_010: LF (112, 138, 40), HF (179, 221, 200)
sample_011: LF (112, 138, 40), HF (179, 221, 200)
sample_012: LF (112, 138, 40), HF (179, 221, 200)
sample_013: LF (112, 138, 40), HF (179, 221, 200)
sample_014: LF (112, 138, 40), HF (179, 221, 200)
sample_015: LF (112, 138, 40), HF (179, 221, 200)
sample_016: LF (112, 138, 40), HF (179, 221, 200)
sample_017: LF (112, 138, 40), HF (179, 221, 200)
sample_018: LF (112, 138, 40), HF (179, 221, 200)

total data loaded: 18


In [20]:
test_data = {}

# If your test files are inside test/low_field (common), set TEST_DIR = "test/low_field"
# TEST_DIR = "test/low_field"

for fname in sorted(os.listdir(TEST_DIR)):
    if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
        continue

    sample_id = fname.replace(
        "_lowfield.nii.gz", "").replace("_lowfield.nii", "")
    sample_id = sample_id.replace(".nii.gz", "").replace(".nii", "")
    path = os.path.join(TEST_DIR, fname)

    vol = load_nifti(path)
    test_data[sample_id] = vol

    print(f"{sample_id}: {vol.shape}")

In [21]:
for sid, d in train_data.items():
    assert np.isfinite(d["low"]).all()
    assert np.isfinite(d["high"]).all()

for sid, vol in test_data.items():
    assert np.isfinite(vol).all()

print("All volumes are finite ✔")

All volumes are finite ✔


## 2. Baseline: Trilinear Upsampling
As a baseline, we upsample the low-field volume to the target shape using trilinear 
interpolation (scipy.ndimage.zoom with order = 1). This simply stretches the image without 
adding any detail, it scores approximately 0.456 on the competition metric.

In [ ]:
TARGET_SHAPE = (179, 221, 200)


def upsample_trilinear(volume: np.ndarray, target_shape=TARGET_SHAPE) -> np.ndarray:
    """
    Trilinear upsampling from (112,138,40) -> (179,221,200).
    Uses scipy.ndimage.zoom with order=1 (trilinear).
    """
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0,
                           neginf=0.0).astype(np.float32)

    zoom_factors = (
        target_shape[0] / volume.shape[0],
        target_shape[1] / volume.shape[1],
        target_shape[2] / volume.shape[2],
    )

    out = zoom(volume, zoom=zoom_factors, order=1)  # trilinear
    # Safety: force exact shape (zoom sometimes lands off by 1 due to rounding)
    out = out[:target_shape[0], :target_shape[1], :target_shape[2]]

    # If any dimension came out smaller, pad to exact shape
    pad_x = target_shape[0] - out.shape[0]
    pad_y = target_shape[1] - out.shape[1]
    pad_z = target_shape[2] - out.shape[2]
    if pad_x > 0 or pad_y > 0 or pad_z > 0:
        out = np.pad(
            out,
            pad_width=((0, max(pad_x, 0)), (0, max(pad_y, 0)),
                       (0, max(pad_z, 0))),
            mode="edge"
        )

    return out.astype(np.float32)

In [23]:
# adjust name if .nii.gz
lf = load_nifti("train/low_field/sample_001_lowfield.nii")
pred = upsample_trilinear(lf)

print("LF:", lf.shape, "-> pred:", pred.shape)

LF: (112, 138, 40) -> pred: (179, 221, 200)


In [ ]:
TEST_DIR = "test/low_field"

predictions = {}

for fname in sorted(os.listdir(TEST_DIR)):
    if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
        continue

    sample_id = fname.replace(
        "_lowfield.nii.gz", "").replace("_lowfield.nii", "")
    path = os.path.join(TEST_DIR, fname)

    lf = load_nifti(path)
    pred_vol = upsample_trilinear(lf)

    predictions[sample_id] = pred_vol
    print(f"{sample_id}: {lf.shape} -> {pred_vol.shape}")

submission_df = create_submission_df(predictions)
submission_df.to_csv("submission.csv", index=False)

print("Wrote submission.csv with rows:", len(submission_df))

sample_019: (112, 138, 40) -> (179, 221, 200)
sample_020: (112, 138, 40) -> (179, 221, 200)
sample_021: (112, 138, 40) -> (179, 221, 200)
sample_022: (112, 138, 40) -> (179, 221, 200)
sample_023: (112, 138, 40) -> (179, 221, 200)
Wrote submission.csv with rows: 1000


## 3. Preprocessing
We normalize both low-field and high-field volumes to [0, 1] using percentile-based 
min-max normalization (clipping at the 1st and 99th percentiles). This ensures both 
volumes are on the same intensity scale, which is critical for the model to learn 
meaningful residuals. A light Gaussian blur (σ = 0.5) is applied to the low-field data 
to reduce noise before upsampling.

In [ ]:
def norm_clip_minmax(vol, p1=1, p99=99):
    v = np.nan_to_num(vol).astype(np.float32)
    lo, hi = np.percentile(v, [p1, p99])
    v = np.clip(v, lo, hi)
    if hi - lo > 1e-8:
        v = (v - lo) / (hi - lo)
    else:
        v = np.zeros_like(v)
    return v

In [ ]:
TARGET_SHAPE = (179, 221, 200)
NUM_SLICES = 200


def upsample(volume: np.ndarray, target_shape=TARGET_SHAPE, order=1) -> np.ndarray:
    """
    order=1 -> trilinear
    order=3 -> tricubic
    """
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0,
                           neginf=0.0).astype(np.float32)

    zoom_factors = (
        target_shape[0] / volume.shape[0],
        target_shape[1] / volume.shape[1],
        target_shape[2] / volume.shape[2],
    )

    out = zoom(volume, zoom=zoom_factors, order=order)

    # Force exact shape (zoom can be off by 1 due to rounding)
    out = out[:target_shape[0], :target_shape[1], :target_shape[2]]

    pad_x = target_shape[0] - out.shape[0]
    pad_y = target_shape[1] - out.shape[1]
    pad_z = target_shape[2] - out.shape[2]
    if pad_x > 0 or pad_y > 0 or pad_z > 0:
        out = np.pad(
            out,
            ((0, max(pad_x, 0)), (0, max(pad_y, 0)), (0, max(pad_z, 0))),
            mode="edge"
        )

    return out.astype(np.float32)


TRAIN_LF_DIR = "train/low_field"
TRAIN_HF_DIR = "train/high_field"

# Collect sample_ids from low_field
lf_files = [f for f in sorted(os.listdir(TRAIN_LF_DIR)) if f.endswith(
    ".nii") or f.endswith(".nii.gz")]
sample_ids = []
for f in lf_files:
    sid = f.replace("_lowfield.nii.gz", "").replace("_lowfield.nii", "")
    sid = sid.replace(".nii.gz", "").replace(".nii", "")
    sample_ids.append(sid)

print("Found train samples:", len(sample_ids))
print("First few:", sample_ids[:5])

solution_rows = []
submission_rows = []

UPSAMPLE_ORDER = 1  # 1 = trilinear, 3 = tricubic
method_name = "tricubic" if UPSAMPLE_ORDER == 3 else "trilinear"

for sid in sample_ids:
    lf_path_candidates = [
        os.path.join(TRAIN_LF_DIR, f"{sid}_lowfield.nii.gz"),
        os.path.join(TRAIN_LF_DIR, f"{sid}_lowfield.nii"),
        os.path.join(TRAIN_LF_DIR, f"{sid}.nii.gz"),
        os.path.join(TRAIN_LF_DIR, f"{sid}.nii"),
    ]
    hf_path_candidates = [
        os.path.join(TRAIN_HF_DIR, f"{sid}_highfield.nii.gz"),
        os.path.join(TRAIN_HF_DIR, f"{sid}_highfield.nii"),
        os.path.join(TRAIN_HF_DIR, f"{sid}.nii.gz"),
        os.path.join(TRAIN_HF_DIR, f"{sid}.nii"),
    ]

    lf_path = next((p for p in lf_path_candidates if os.path.exists(p)), None)
    hf_path = next((p for p in hf_path_candidates if os.path.exists(p)), None)

    if lf_path is None or hf_path is None:
        raise FileNotFoundError(
            f"Could not find pair for {sid}.\nLF candidates: {lf_path_candidates}\nHF candidates: {hf_path_candidates}")

    lf = load_nifti(lf_path)
    hf = load_nifti(hf_path)

    lf = norm_clip_minmax(lf)
    lf = gaussian_filter(lf, sigma=0.5)
    pred = upsample(lf, order=UPSAMPLE_ORDER)

    if hf.shape != TARGET_SHAPE:
        raise ValueError(f"{sid}: expected HF {TARGET_SHAPE}, got {hf.shape}")
    if pred.shape != TARGET_SHAPE:
        raise ValueError(
            f"{sid}: expected pred {TARGET_SHAPE}, got {pred.shape}")

    for z in range(NUM_SLICES):
        row_id = f"{sid}_slice_{z:03d}"
        solution_rows.append({
            "row_id": row_id,
            "ground_truth": slice_to_base64(hf[:, :, z]),
            "Usage": "Public"
        })
        submission_rows.append({
            "row_id": row_id,
            "prediction": slice_to_base64(pred[:, :, z])
        })

solution_df = pd.DataFrame(solution_rows)
submission_df = pd.DataFrame(submission_rows)

print("Solution rows:", len(solution_df),
      "Submission rows:", len(submission_df))

# baseline_score = metric.score(solution_df, submission_df, "row_id")
# print(f"{method_name.capitalize()} baseline score:", baseline_score)

Found train samples: 18
First few: ['sample_001', 'sample_002', 'sample_003', 'sample_004', 'sample_005']
Solution rows: 3600 Submission rows: 3600


## 4. Dataset: 3D Patch Extraction
Since the full volumes are too large to fit in GPU memory, we extract random 3D patches 
(64×64×32) from each volume during training. The dataset returns:
- **x**: a patch from the bicubic-upsampled baseline
- **y**: the corresponding residual (high-field patch minus baseline patch)

The model learns to predict this residual; what needs to be "added" to the blurry 
baseline to make it look like the sharp high-field image.

In [ ]:
TARGET_SHAPE = (179, 221, 200)


def norm_clip_minmax(vol, p1=1, p99=99):
    v = np.nan_to_num(vol).astype(np.float32)
    lo, hi = np.percentile(v, [p1, p99])
    v = np.clip(v, lo, hi)
    if hi - lo > 1e-8:
        v = (v - lo) / (hi - lo)
    else:
        v = np.zeros_like(v)
    return v


def upsample(vol, target_shape=TARGET_SHAPE, order=3):
    from scipy.ndimage import zoom
    zoom_factors = (
        target_shape[0] / vol.shape[0],
        target_shape[1] / vol.shape[1],
        target_shape[2] / vol.shape[2],
    )
    out = zoom(vol, zoom=zoom_factors, order=order)
    out = out[:target_shape[0], :target_shape[1], :target_shape[2]]
    pad = (target_shape[0]-out.shape[0],
           target_shape[1]-out.shape[1],
           target_shape[2]-out.shape[2])
    if any(p > 0 for p in pad):
        out = np.pad(out,
                     ((0, max(pad[0], 0)),
                      (0, max(pad[1], 0)),
                      (0, max(pad[2], 0))),
                     mode="edge")
    return out.astype(np.float32)


class Residual3DPatchDataset(Dataset):
    """
    Returns:
      x: baseline patch  (1, px, py, pz)
      y: residual patch  (1, px, py, pz)  where residual = HF - baseline
    """

    def __init__(
        self,
        sample_ids,
        train_lf_dir,
        train_hf_dir,
        patch_size=(96, 96, 64),
        patches_per_volume=64,
        upsample_order=3,
        denoise_sigma=0.5,
        seed=42,
    ):
        self.sample_ids = list(sample_ids)
        self.patch_size = patch_size
        self.patches_per_volume = patches_per_volume
        self.rng = np.random.default_rng(seed)

        self.baselines = {}
        self.residuals = {}

        for sid in self.sample_ids:
            lf_path = self._find_path(train_lf_dir, sid, low=True)
            hf_path = self._find_path(train_hf_dir, sid, low=False)

            lf = load_nifti(lf_path)
            hf = load_nifti(hf_path).astype(np.float32)

            lf = norm_clip_minmax(lf)
            lf = gaussian_filter(lf, sigma=denoise_sigma)
            hf = norm_clip_minmax(hf)

            baseline = upsample(lf, order=upsample_order)
            residual = hf - baseline

            self.baselines[sid] = baseline
            self.residuals[sid] = residual

        self.length = len(self.sample_ids) * self.patches_per_volume

    def _find_path(self, folder, sid, low=True):
        suffix = "lowfield" if low else "highfield"
        candidates = [
            os.path.join(folder, f"{sid}_{suffix}.nii.gz"),
            os.path.join(folder, f"{sid}_{suffix}.nii"),
            os.path.join(folder, f"{sid}.nii.gz"),
            os.path.join(folder, f"{sid}.nii"),
        ]
        path = next((p for p in candidates if os.path.exists(p)), None)
        if path is None:
            raise FileNotFoundError(f"Missing file for {sid} in {folder}")
        return path

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        vidx = idx // self.patches_per_volume
        sid = self.sample_ids[vidx]

        base = self.baselines[sid]
        res = self.residuals[sid]

        px, py, pz = self.patch_size
        X, Y, Z = base.shape

        x0 = self.rng.integers(0, X - px + 1)
        y0 = self.rng.integers(0, Y - py + 1)
        z0 = self.rng.integers(0, Z - pz + 1)

        x_patch = base[x0:x0+px, y0:y0+py, z0:z0+pz]
        y_patch = res[x0:x0+px, y0:y0+py, z0:z0+pz]

        x_t = torch.from_numpy(x_patch[None, ...])
        y_t = torch.from_numpy(y_patch[None, ...])

        return x_t, y_t

## 5. Cross-Validation Setup
With only 18 training volumes, we use 6-fold cross-validation (15 train / 3 validation 
per fold) to get a reliable estimate of model performance and avoid overfitting to any 
particular subset of brains.

In [ ]:
TRAIN_LF_DIR = "train/low_field"

# Get sample IDs
lf_files = [f for f in sorted(os.listdir(TRAIN_LF_DIR)) if f.endswith(
    ".nii") or f.endswith(".nii.gz")]
sample_ids = []
for f in lf_files:
    sid = f.replace("_lowfield.nii.gz", "").replace("_lowfield.nii", "")
    sid = sid.replace(".nii.gz", "").replace(".nii", "")
    sample_ids.append(sid)

# ✅ FIX: ensure numpy array with string dtype
sample_ids = np.array(sorted(sample_ids), dtype=str)

print("Total volumes:", len(sample_ids), "| dtype:", sample_ids.dtype)

kf = KFold(n_splits=6, shuffle=True, random_state=42)

folds = []
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(sample_ids), start=1):
    # ✅ FIX: make sure indices are int arrays (extra-safe)
    train_idx = np.asarray(train_idx, dtype=int)
    val_idx = np.asarray(val_idx, dtype=int)

    train_ids = sample_ids[train_idx]
    val_ids = sample_ids[val_idx]

    folds.append((train_ids, val_ids))
    print(
        f"Fold {fold_idx}: train = {len(train_ids)} val = {len(val_ids)} | val_ids = {list(val_ids)}")

Total volumes: 18 | dtype: <U10
Fold 1: train = 15 val = 3 | val_ids = [np.str_('sample_001'), np.str_('sample_002'), np.str_('sample_009')]
Fold 2: train = 15 val = 3 | val_ids = [np.str_('sample_004'), np.str_('sample_006'), np.str_('sample_014')]
Fold 3: train = 15 val = 3 | val_ids = [np.str_('sample_012'), np.str_('sample_016'), np.str_('sample_017')]
Fold 4: train = 15 val = 3 | val_ids = [np.str_('sample_003'), np.str_('sample_010'), np.str_('sample_018')]
Fold 5: train = 15 val = 3 | val_ids = [np.str_('sample_005'), np.str_('sample_008'), np.str_('sample_013')]
Fold 6: train = 15 val = 3 | val_ids = [np.str_('sample_007'), np.str_('sample_011'), np.str_('sample_015')]


## 6. Model Architecture & Training
**Architecture:** A 3D ResNet with 4 residual blocks, each containing two 3D convolutions 
with InstanceNorm and LeakyReLU activations. The model has 16 feature channels and 
predicts the residual to add to the baseline.

**Training details:**
- **Loss:** L1 (mean absolute error) on the predicted residual
- **Optimizer:** AdamW (lr = 2e-4, weight_decay = 1e-4)
- **Epochs:** 25 per fold
- **Gradient clipping:** max norm = 1.0
- **Batch size:** 1 (due to 3D patch memory requirements)

In [ ]:
TRAIN_LF_DIR = "train/low_field"
TRAIN_HF_DIR = "train/high_field"

PATCH_SIZE = (64, 64, 32)
TRAIN_PATCHES_PER_VOL = 16
VAL_PATCHES_PER_VOL = 8
BATCH_SIZE = 1
EPOCHS = 25
LR = 2e-4
WEIGHT_DECAY = 1e-4


def make_datasets(train_ids, val_ids):
    train_ds = Residual3DPatchDataset(
        train_ids,
        TRAIN_LF_DIR,
        TRAIN_HF_DIR,
        patch_size=PATCH_SIZE,
        patches_per_volume=TRAIN_PATCHES_PER_VOL,
        upsample_order=3,
        denoise_sigma=0.5,
        seed=42,
    )
    val_ds = Residual3DPatchDataset(
        val_ids,
        TRAIN_LF_DIR,
        TRAIN_HF_DIR,
        patch_size=PATCH_SIZE,
        patches_per_volume=VAL_PATCHES_PER_VOL,
        upsample_order=3,
        denoise_sigma=0.5,
        seed=999,
    )
    return train_ds, val_ds


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    losses = []
    for x, y in loader:
        # x, y = x.to(device), y.to(device)
        x, y = x.float().to(device), y.float().to(device)
        pred = model(x)
        loss = torch.nn.functional.l1_loss(pred, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    losses = []
    for x, y in loader:
        # x, y = x.to(device), y.to(device)
        x, y = x.float().to(device), y.float().to(device)
        pred = model(x)
        loss = torch.nn.functional.l1_loss(pred, y)
        losses.append(loss.item())
    return float(np.mean(losses))


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Device:", device)

# -----------------------------
# ResNet-style 3D residual model
# -----------------------------


class ResBlock3D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(ch, ch, 3, padding=1),
            nn.InstanceNorm3d(ch),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(ch, ch, 3, padding=1),
            nn.InstanceNorm3d(ch),
        )
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        return self.act(x + self.block(x))


class ResNet3D_SR(nn.Module):
    """Predicts residual to add to the baseline."""

    def __init__(self, n_feats=16, n_blocks=4):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv3d(1, n_feats, 3, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock3D(n_feats)
                                  for _ in range(n_blocks)])
        self.tail = nn.Conv3d(n_feats, 1, 3, padding=1)

    def forward(self, x):
        x = self.head(x)
        x = self.body(x)
        return self.tail(x)

# -----------------------------
# 6-fold CV loop (folds already contains train_ids, val_ids)
# -----------------------------


fold_best_scores = []

for fold_idx, (train_ids, val_ids) in enumerate(folds, start=1):
    print(f"\n=== Fold {fold_idx}/6 ===")
    print("Train volumes:", len(train_ids), "| Val volumes:", len(val_ids))

    train_ds, val_ds = make_datasets(train_ids, val_ids)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0)

    model = ResNet3D_SR(n_feats=16, n_blocks=4).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = float("inf")
    best_path = f"best_fold_{fold_idx}.pt"

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = validate(model, val_loader, device)
        print(
            f"Fold {fold_idx} | Epoch {epoch:02d} | train L1 = {train_loss:.4f} | val L1 = {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), best_path)

    print(f"✅ Fold {fold_idx} best val: {best_val:.4f} (saved {best_path})")
    fold_best_scores.append(best_val)

mean_score = float(np.mean(fold_best_scores))
std_score = float(np.std(fold_best_scores))
print("\n=== CV Summary ===")
print("Fold best vals:", [round(s, 4) for s in fold_best_scores])
print(f"Mean ± Std: {mean_score:.4f} ± {std_score:.4f}")

Device: mps

=== Fold 1/6 ===
Train volumes: 15 | Val volumes: 3
Fold 1 | Epoch 01 | train L1 = 0.1820 | val L1 = 0.1682
Fold 1 | Epoch 02 | train L1 = 0.1423 | val L1 = 0.1516
Fold 1 | Epoch 03 | train L1 = 0.1367 | val L1 = 0.1507
Fold 1 | Epoch 04 | train L1 = 0.1352 | val L1 = 0.1712
Fold 1 | Epoch 05 | train L1 = 0.1300 | val L1 = 0.1565
Fold 1 | Epoch 06 | train L1 = 0.1276 | val L1 = 0.1542
Fold 1 | Epoch 07 | train L1 = 0.1322 | val L1 = 0.1203
Fold 1 | Epoch 08 | train L1 = 0.1279 | val L1 = 0.1176
Fold 1 | Epoch 09 | train L1 = 0.1263 | val L1 = 0.1439
Fold 1 | Epoch 10 | train L1 = 0.1282 | val L1 = 0.1529
Fold 1 | Epoch 11 | train L1 = 0.1301 | val L1 = 0.1375
Fold 1 | Epoch 12 | train L1 = 0.1268 | val L1 = 0.1527
Fold 1 | Epoch 13 | train L1 = 0.1208 | val L1 = 0.1391
Fold 1 | Epoch 14 | train L1 = 0.1225 | val L1 = 0.1360
Fold 1 | Epoch 15 | train L1 = 0.1226 | val L1 = 0.1397
Fold 1 | Epoch 16 | train L1 = 0.1237 | val L1 = 0.1384
Fold 1 | Epoch 17 | train L1 = 0.1202 |

## 7. Inference & Submission
For our initial submission, we use tricubic interpolation (order=3) directly on the raw 
low-field volumes. While our trained model showed learning on the residuals, the intensity 
rescaling between normalized model outputs and the raw ground truth remains a challenge 
we are continuing to investigate for the next submission deadline.

In [ ]:
TEST_DIR = "test/low_field"
predictions = {}

for fname in sorted(os.listdir(TEST_DIR)):
    if not (fname.endswith(".nii") or fname.endswith(".nii.gz")):
        continue
    sample_id = fname.replace(
        "_lowfield.nii.gz", "").replace("_lowfield.nii", "")
    path = os.path.join(TEST_DIR, fname)

    lf_vol = load_nifti(path)
    pred = upsample(lf_vol.astype(np.float32), order=3)
    predictions[sample_id] = pred
    print(f"{sample_id}: done, shape {pred.shape}")

submission_df = create_submission_df(predictions)
submission_df.to_csv("submission.csv", index=False)
print(f"Wrote submission.csv with {len(submission_df)} rows")

sample_019: done
sample_020: done
sample_021: done
sample_022: done
sample_023: done
Wrote submission.csv with 1000 rows
